# Construindo um Agente de RAG Especializado com OpenAI e LangChain

Este notebook guiará você no desenvolvimento de um sistema de **RAG (Retrieval-Augmented Generation)**. Você aprenderá a criar um assistente virtual inteligente capaz de ler um documento PDF técnico (o manual de um Nobreak) e responder a dúvidas de clientes de forma precisa, utilizando uma base de dados de vetor para busca de informações e mantendo o histórico de conversação.

---

### Objetivos de Aprendizagem
* Entender o conceito de **RAG (Geração Aumentada de Recuperação)** e por que ele é usado para evitar alucinações em LLMs.
* Aprender a extrair e processar textos de arquivos PDF com o `PyPDFLoader`.
* Compreender o que são **Embeddings** e como criar um banco de vetores utilizando o **FAISS**.
* Integrar histórico de mensagens à conversação utilizando o LangChain para criar uma experiência de chat real.

In [1]:
!pip install -U langchain langchain-community langchain-core langchain-openai faiss-cpu pypdf

## Passo 1: Preparação do Ambiente e Bibliotecas

Antes de começar, precisamos preparar nossa máquina virtual com as ferramentas necessárias. O ecossistema do **LangChain** é modular. Aqui está o porquê de cada biblioteca que estamos instalando:

* **`langchain`**: O framework principal que nos ajuda a encadear as chamadas do modelo com outras ferramentas.
* **`langchain-community`**: Contém integrações feitas pela comunidade (como carregadores de PDF e bancos de dados).
* **`langchain-core`**: A base de dados estruturada e interfaces que o LangChain utiliza.
* **`langchain-openai`**: A integração específica para utilizarmos os modelos e embeddings da OpenAI.
* **`faiss-cpu`**: Nosso banco de dados vetorial de alta performance (Facebook AI Similarity Search) para buscar trechos de texto semelhantes.
* **`pypdf`**: A biblioteca que o Python usa nos bastidores para ler arquivos PDF.

In [6]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.messages.system import SystemMessage

## Passo 2: Importação das Dependências

Aqui, importamos os blocos de construção do nosso agente.

* **`PyPDFLoader`**: Carrega e extrai o texto do PDF.
* **`OpenAIEmbeddings`**: Transforma palavras/frases em números (vetores) que capturam o significado semântico.
* **`FAISS`**: Armazena esses vetores e permite uma busca rápida por proximidade de significado.
* **`ChatOpenAI`**: Interface para nos comunicarmos com o modelo de linguagem (GPT-4o).
* **`InMemoryChatMessageHistory`** & **`RunnableWithMessageHistory`**: Ferramentas para o nosso agente lembrar do contexto das perguntas anteriores na mesma sessão.

In [ ]:
OPENAI_API_KEY = 'SUA_CHAVE_KEY' # Lembre-se de usar a variavel de ambiente

## Passo 3: Configuração da API Key

Para interagir com os modelos de IA da OpenAI, precisamos de uma chave de API válida.

> 💡 **Para pensar (Dica de Segurança):** No código atual, a chave de API está definida diretamente como uma string fictícia `'SUA_CHAVE_KEY'`. Em ambientes de produção ou repositórios públicos, nunca exponha suas chaves! O ideal no Google Colab é utilizar o gerenciador de segredos (ícone de chave na barra lateral esquerda) e acessá-la de forma segura usando: `from google.colab import userdata; os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')`.

In [ ]:
loader = PyPDFLoader("manual_avancado_nobreak_fxp2000.pdf")
documents = loader.load()

## Passo 4: Carregamento do Documento Técnico

Utilizamos o `PyPDFLoader` para ler o arquivo PDF do manual do Nobreak.

* **O porquê:** Modelos como o GPT não conhecem as especificidades do nobreak FXP2000 por padrão. Precisamos municiar nosso sistema com este documento técnico.
* **O como:** O carregador lê o arquivo do disco local do Colab e quebra seu conteúdo em uma lista de objetos `Document`, onde cada página vira um bloco contendo o texto extraído e os metadados (como o número da página).

In [ ]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents, embeddings)
retriver = vectorstore.as_retriever()

## Passo 5: Vetorização e Banco de Vetores (Embeddings & FAISS)

Esta é a mágica do RAG.

* **O que são Embeddings?** São representações numéricas (listas de números chamadas vetores) de palavras ou frases. Computadores não entendem sentimentos ou manuais, mas entendem matemática. Frases com significados parecidos terão vetores muito próximos no espaço matemático.
* **O que é o FAISS?** É um banco de dados especializado em procurar quais vetores são mais parecidos com o vetor da pergunta do usuário.
* **Como funciona:** O código pega os textos extraídos do manual, gera os números (embeddings) correspondentes usando a OpenAI e os armazena no FAISS. O `as_retriever()` transforma essa base em uma ferramenta de busca automática.

> 💡 **Para pensar (Chunking):** No código atual, enviamos os documentos inteiros sem fazer uma divisão prévia em partes menores (conhecida como *chunking*). Para PDFs longos, enviar a página inteira pode exceder o limite de contexto do modelo ou trazer excesso de ruído. Usar o `RecursiveCharacterTextSplitter` antes de criar o banco de dados vetorial costuma aumentar muito a precisão do sistema.

In [ ]:
chat = ChatOpenAI(model='gpt-4o', temperatura=0)

## Passo 6: Inicialização do Modelo de Chat

Instanciamos o modelo de linguagem **GPT-4o** da OpenAI para processar as informações recuperadas do PDF e formatar a resposta ideal para o usuário.

> 💡 **Para pensar (Parâmetros):** O código usa a propriedade `temperatura=0` (com um pequeno erro de digitação de português: `temperatura` em vez do parâmetro correto em inglês `temperature`). Definir a temperatura como `0` é excelente para tarefas de RAG e suporte técnico porque força o modelo a ser mais preciso, factual e menos criativo, evitando que ele invente dados sobre o nobreak.

In [ ]:
system_message = SystemMessage(content='''
Você é um assistente virtual especializado no atendimento ao cliente no nobreak FCP-2000.
Responda claramente perguntas tecnicas, funcionalidades, garantia, manutenção, atualização e suporte técnico.
Se a pergunte for inrrelevante, responda educadamente recusando a pergunta.
''')

## Passo 7: Definição da Persona do Assistente (System Message)

Aqui, criamos uma instrução de sistema que diz ao modelo exatamente quem ele é e como deve se comportar.

* **O porquê:** Isso garante que o modelo mantenha a postura de suporte técnico e recuse educadamente responder a perguntas aleatórias que não tenham relação com o manual do nobreak.

> [VERIFICAR]: No código atual, a variável `system_message` é criada mas ela não está sendo injetada ou passada como parâmetro na construção da `ConversationalRetrievalChain` abaixo. Para que ela tenha efeito prático, ela precisaria ser configurada como parte do prompt do sistema na pipeline de QA.

In [ ]:
# memoria por sessaão "chat_id"
store = {}

def get_session_history(session_id: str)
  if session_id not in store:
    store[session_id] = InMemoryChatMessageHistory()
  return store[session_id]

qa_chain = ConversationalRetrievalChain.from_llm(
    llm = chat,
    retrival = retriver,
    return_souce_documents = False
)

chain_with_history = RunnableWithMessageHistory(
    qa_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

## Passo 8: Criação da Cadeia de Conversação com Histórico

Aqui construímos o fluxo lógico do chatbot.

* **`ConversationalRetrievalChain`**: Essa cadeia conecta o modelo de linguagem (`chat`) e o recuperador de dados vetoriais (`retriver`). Quando o usuário faz uma pergunta, ela pesquisa trechos relevantes no PDF, junta com a pergunta do usuário e entrega tudo mastigado para a LLM responder.
* **`RunnableWithMessageHistory`**: Adiciona a capacidade de lembrar do contexto. Sem isso, cada pergunta feita seria tratada de forma isolada, impedindo o usuário de dizer coisas como "E como eu resolvo isso?" logo após uma resposta.

> 💡 **Para pensar (Erros de digitação):**
* Na definição de `get_session_history`, está faltando os dois pontos (`:`) no final da linha `def get_session_history(session_id: str)`.
* Na chamada `ConversationalRetrievalChain.from_llm`, o parâmetro correto do recuperador é `retriever` (com 'e') e não `retrival`, e o parâmetro de fontes de documentos é `return_source_documents` e não `return_souce_documents`.

In [ ]:
# Testes
session_id = "sessao1"
while True:
  pergunta = input("Você: ")
  if pergunta.lower() in ["sair", "quit", "exit"]:
    print("vlw, Até logo")
  resposta = chain_with_history.invoke(
      {"question": pergunta},
      config={"configurable":{"session_id": session_id}}
  )
  print("Assistente:", resposta["answer"])

## Passo 9: Loop de Teste Interativo

Este trecho de código cria uma interface interativa diretamente no seu terminal do Colab para conversar com o assistente.

* **O como:** O loop `while True` continuará rodando infinitamente, coletando as perguntas do usuário (`input`), até que a palavra chave de parada seja digitada (como `sair`, `quit` ou `exit`). O assistente recebe a pergunta vinculada ao ID da sessão (`session_id`), o que permite que ele mantenha a conversa contínua.

---

## Erros Comuns e Dicas de Resolução

1. **`ModuleNotFoundError` no import de `langchain_classic.chains`**: O LangChain passou por muitas reestruturações. Se a importação clássica falhar mesmo após reiniciar sua sessão, o ideal é usar as novas APIs do LangChain Core (`create_retrieval_chain` de `langchain.chains`).
2. **`NameError: name 'chat' is not defined`**: Certifique-se de executar todas as células anteriores em ordem sequencial. Se você reiniciar o ambiente de execução, todas as variáveis armazenadas na memória são limpas e você precisará executar o notebook de cima para baixo novamente.
3. **`SyntaxError` no `def get_session_history`**: O Python exige o caractere `:` ao final da assinatura de qualquer função. Adicione `:` na linha `def get_session_history(session_id: str)`.

---

## Resumo Final

Neste notebook você aprendeu:
* **RAG na prática**: Como dar acesso a um conhecimento privado (um PDF) para um modelo de inteligência artificial de forma dinâmica.
* **Processo de Vetorização**: Extrair textos, transformá-los em vetores numéricos de significado (Embeddings) e armazenar em um banco vetorial especializado (FAISS).
* **Manutenção de Contexto**: Como conectar memória à execução de uma chain para que o bot entenda conversas continuadas.

---

## Exercícios Propostos

1. **[Fácil]** Adicione uma mensagem de boas-vindas amigável para o usuário assim que o loop de testes iniciar, antes de pedir a primeira entrada.
2. **[Médio]** Modifique a configuração do modelo de chat `ChatOpenAI` para experimentar valores de `temperature` maiores (como `0.7`) e observe se o assistente começa a responder de forma mais criativa ou se começa a cometer erros sobre os dados técnicos do nobreak.
3. **[Desafiador]** Implemente um divisor de texto (`RecursiveCharacterTextSplitter` do LangChain) entre a extração do PDF (`loader.load()`) e a criação do banco de dados FAISS. Configure o tamanho dos fragmentos (*chunk_size*) para 1000 caracteres com sobreposição (*chunk_overlap*) de 200 caracteres, e analise o impacto no desempenho das respostas.

---

## Para Ir Além

* Pesquise sobre **LCEL (LangChain Expression Language)**: a nova forma recomendada e extremamente poderosa para construir fluxos de dados e chains personalizadas de IA no LangChain.
* Leia sobre as diferentes estratégias de **Chunking** e como quebrar documentos longos pode alterar drasticamente o custo e a assertividade do seu sistema RAG.
* Estude sobre **Avaliação de RAG (RAG Triad)** para medir matematicamente se o seu recuperador está trazendo documentos úteis e se a sua LLM está sendo fiel a esses documentos.